# REMI

Recall visualization and planner training.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import gc
import os
import random

import faiss
import numpy as np
from omegaconf import OmegaConf
import torch

from remi.model import HCMEC
from remi.planner import Planner
from remi.rtgym_lite.dataclass import AgentState
from remi.utils.visualization import (
    configure_plot_style,
    plot_planning_paths,
    plot_ratemaps,
    plot_recall_error,
    plot_recall_manifolds,
    plot_recall_snapshots,
    plot_training_loss,
    select_recall_trial,
)


SEED = 42
PLANNER_EPOCHS = 16
PLANNER_STEPS_PER_EPOCH = 20
PLANNER_STEPS = PLANNER_EPOCHS * PLANNER_STEPS_PER_EPOCH

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required.")
DEVICE = torch.device("cuda")


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


configure_plot_style()
seed_everything()


## 1. Load HC–MEC and ratemaps

In [ ]:
ARTIFACT_DIR = Path(os.environ.get("REMI_ARTIFACT_DIR", "artifacts"))
if not (ARTIFACT_DIR / "hcmec.pt").exists() or not (ARTIFACT_DIR / "ratemaps.npz").exists():
    raise FileNotFoundError(f"Missing model artifacts in {ARTIFACT_DIR}.")

checkpoint = ARTIFACT_DIR / "hcmec.pt"
checkpoint_data = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
model = HCMEC.from_state_dict(checkpoint_data["model"])
model.to(DEVICE)
model.eval()

with np.load(ARTIFACT_DIR / "ratemaps.npz") as data:
    arena_map = data["arena_map"].copy()
    visit_counts = data["visit_counts"].copy()
    learned_ratemaps = {
        key: data[f"learned_{key}"].copy() for key in ("wsm", "gc", "hpc")
    }
    ground_truth_ratemaps = {
        key: data[f"target_{key}"].copy() for key in ("wsm", "gc")
    }


class RatemapDecoder:
    def __init__(self, rate_map, counts):
        valid = (counts > 0) & np.isfinite(rate_map).all(axis=0)
        self.coordinates = np.argwhere(valid).astype(np.float32)
        vectors = np.moveaxis(rate_map, 0, -1)[valid].astype(np.float32)
        self.index = faiss.IndexFlatL2(vectors.shape[1])
        self.index.add(np.ascontiguousarray(vectors))

    def __call__(self, response):
        if torch.is_tensor(response):
            response = response.detach().cpu().numpy()
        shape = response.shape[:-1]
        queries = np.ascontiguousarray(response.reshape(-1, response.shape[-1]), dtype=np.float32)
        _, nearest = self.index.search(queries, 1)
        return self.coordinates[nearest[:, 0]].reshape(*shape, 2)


decoders = {
    key: RatemapDecoder(rate_map, visit_counts)
    for key, rate_map in learned_ratemaps.items()
}
print(f"device: {DEVICE}")
print(f"artifacts: {ARTIFACT_DIR}")
print(f"hidden units: {model.size}")


## 2. Ratemaps

In [ ]:
plot_ratemaps(learned_ratemaps, ground_truth_ratemaps)

## 3. Recall

In [ ]:
seed_everything(SEED + 1)
n_queries = 512
query_positions = model.gym.arena.generate_random_pos(n_queries)
query_states = AgentState(coord=query_positions)
smc_cue = model.gym.agent.get_response(query_states, str_filter="wsm")

model.eval()
with torch.no_grad():
    init, inputs = model.generate_query_data(
        torch.tensor(smc_cue, dtype=torch.float32, device=DEVICE),
        duration=5,
    )
    _, recalled = model(inputs, init)

recall_coordinates = {}
recall_distances = {}
for key in ("wsm", "gc", "hpc"):
    decoded = decoders[key](recalled[model.region_dict[key]])
    recall_coordinates[key] = decoded
    recall_distances[key] = np.linalg.norm(
        decoded - query_positions[:, None, :], axis=-1
    )

time_s = np.arange(recall_distances["wsm"].shape[1]) * 0.05
for key, label in (("wsm", "SMC"), ("gc", "GC"), ("hpc", "HPC")):
    mean = recall_distances[key].mean(axis=0)
    print(
        f"{label}: start={mean[0]:.2f} cm, "
        f"best={mean.min():.2f} cm, final={mean[-1]:.2f} cm"
    )

plot_recall_error(recall_distances, time_s)


In [ ]:
trial = select_recall_trial(recall_distances)

plot_recall_snapshots(
    arena_map,
    query_positions,
    recall_coordinates,
    recall_distances,
    time_s,
    trial,
)
plot_recall_manifolds(
    learned_ratemaps,
    recalled,
    model.region_dict,
    query_positions,
    trial,
    seed=SEED,
)


## 4. Train planner

In [ ]:
planner_cfg = OmegaConf.create({
    "loss_keys": ["gc"],
    "init_keys": ["gc", "wsm"],
    "planner_keys": ["gc"],
    "control_keys": ["dir", "vel"],
    "hidden_dim": 1024,
    "alpha": 0.2,
    "learn_alpha": True,
    "loss_cfg": {
        "vel": {"lambda": 0},
        "vel_std": {"lambda": 0},
        "max_vel": {"lambda": 0},
        "enc": {"lambda": 0},
        "plan": {"lambda": 1, "alpha": 0, "beta": 0},
        "norm": {"lambda": 0},
        "eig": {"lambda": 0},
        "goal": {"lambda": 0},
    },
})

seed_everything(SEED + 2)
planner = Planner(planner_cfg, checkpoint, device=DEVICE)
planner.to(DEVICE)
planner_optimizer = torch.optim.AdamW(
    (p for p in planner.parameters() if p.requires_grad), lr=1e-3
)
planner_loss = []

for step in range(PLANNER_STEPS):
    init, inputs, target, _ = planner.generate_data(
        duration=1, batch_size=512, pause_prob=0.2
    )
    _, states = planner(inputs, init)
    loss, _ = planner.compute_loss(states, target)
    planner_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(planner.parameters(), 1.0)
    planner_optimizer.step()
    planner_loss.append(float(loss.detach()))
    del init, inputs, target, states, loss, _
    gc.collect()
    if (step + 1) % PLANNER_STEPS_PER_EPOCH == 0:
        print(f"planner {step + 1:3d}/{PLANNER_STEPS}: loss={np.mean(planner_loss[-20:]):.3f}")

plot_training_loss(planner_loss, "Planner training")


## 5. Planning

In [ ]:
seed_everything(SEED + 3)
decoded_batches = {"gc": [], "wsm": []}
reference_batches = []

with torch.no_grad():
    for _ in range(16):
        init, inputs, target, reference = planner.generate_data(
            duration=10, batch_size=256, pause_prob=0
        )
        outputs, planned_states = planner(inputs, init)
        reference_batches.append(reference.coord)
        for key in decoded_batches:
            decoded_batches[key].append(decoders[key](planner.parse_response(planned_states, key)))
        del init, inputs, target, outputs, planned_states
        gc.collect()

decoded_plan = {key: np.concatenate(value) for key, value in decoded_batches.items()}
reference_paths = np.concatenate(reference_batches)
goals = reference_paths[:, -1]

for key, label in (("gc", "GC"), ("wsm", "SMC")):
    final_distance = np.linalg.norm(decoded_plan[key][:, -1] - goals, axis=-1)
    print(
        f"{label}: final distance={final_distance.mean():.2f} ± "
        f"{final_distance.std():.2f} cm; success@10cm="
        f"{(final_distance < 10).mean():.2%}"
    )

plot_planning_paths(reference_paths, decoded_plan)
